# Tommy Award hustle data — MLP (Keras) application

This notebook applies the **feedforward (dense) MLP** workflow from `CH10.ipynb` to `Tommy_Award_Player_Game_Table_hustle.csv`.

**Target:** `y` — whether this player is the per-game “hustle” winner (1) or not (0).

> Requires **TensorFlow / Keras** in your environment (same as CH10), e.g. `pip install tensorflow`.

## What we did in `CH10.ipynb` (neural network methods)

1. **Theory (ISLR Ch.10):** A feedforward network with one or more **hidden** layers. Each unit applies an **activation** (e.g. sigmoid, ReLU) to a linear combination of inputs; the last layer for multi-class problems uses **softmax** to output class probabilities.
2. **Keras MLP (tabular toy example):** A `Sequential` model with `Input(16)` → two `Dense(32, activation="sigmoid")` layers → `Flatten` → `Dense(5, activation="softmax")` for **5-class** classification, trained with `sparse_categorical_crossentropy` and the **Adam** optimizer, using **sparse** integer labels and metrics like accuracy.
3. **CNN (MNIST):** For **image** data, a small **Conv2D** → **MaxPooling2D** stack, **Dropout** before the final dense layer, **one-hot** labels, `categorical_crossentropy`, and **SGD** — reaching high test **accuracy** on digit classification.

Here we use the **MLP (dense) pattern**, not a CNN, because the hustle table is **row-per-player per-game** tabular data, not images. We adapt the output to **binary** classification (`sigmoid` + `binary_crossentropy`) and **standardize** numeric inputs before training.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearnmport Simple.impute iImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
import keras
from keras import layers

RANDOM = 42
np.random.seed(RANDOM)
tf.random.set_seed(RANDOM)
print(keras.__version__)

SyntaxError: invalid syntax (1549834921.py, line 5)

In [ ]:
from pathlib import Path

candidates = [
    Path("Tommy_Award_Player_Game_Table_hustle.csv"),
    Path("Notes") / "Tommy_Award_Player_Game_Table_hustle.csv",
]
DATA_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place Tommy_Award_Player_Game_Table_hustle.csv next to this notebook, "
        "or run the notebook with cwd set to the project root."
    )
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df)}  Columns: {len(df.columns)}  from {DATA_PATH}")
print(df["y"].value_counts())

In [ ]:
# Numeric features only; drop IDs and the label (mirrors a clean tabular MLP setup)
ID_LIKE = {"gameId", "teamId", "personId", "GAME_ID", "jerseyNum"}
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in ID_LIKE and c != "y"]
X = df[feature_cols]
y = df["y"].astype(int).values
print(f"Using {len(feature_cols)} numeric features (IDs and y excluded).")
print(feature_cols[:8], "...", feature_cols[-4:])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM, stratify=y
)
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Class imbalance: weight the positive class (same idea as sklearn class_weight)
n0, n1 = (y == 0).sum(), (y == 1).sum()
class_weight = {0: 1.0, 1: n0 / n1}
print(f"class_weight for y=1: {class_weight[1]:.3f}")

In [ ]:
n_features = X_train.shape[1]

# CH10-style Sequential MLP: dense stack → final activation suited to the task (binary here)
mlp = keras.Sequential(name="hustle_mlp")
mlp.add(layers.Input(shape=(n_features,)))
mlp.add(layers.Dense(64, activation="relu", name="dense_1"))
mlp.add(layers.Dropout(0.3, name="dropout_1"))
mlp.add(layers.Dense(32, activation="relu", name="dense_2"))
mlp.add(layers.Dense(1, activation="sigmoid", name="prob_hustle_winner"))

mlp.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        keras.metrics.AUC(name="auc"),
        keras.metrics.BinaryAccuracy(name="acc"),
    ],
)
mlp.build(input_shape=(None, n_features))
mlp.summary()

In [ ]:
EPOCHS = 40
BATCH = 64
early = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=6, restore_best_weights=True
)
history = mlp.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH,
    class_weight=class_weight,
    callbacks=[early],
    verbose=1,
)

In [ ]:
p_test = mlp.predict(X_test, verbose=0).ravel()
y_hat = (p_test >= 0.5).astype(int)
print("Test accuracy:", float(accuracy_score(y_test, y_hat)))
print("Test ROC-AUC:", float(roc_auc_score(y_test, p_test)))
print("\nConfusion matrix (rows: true, cols: pred 0,1):\n", confusion_matrix(y_test, y_hat))
print("\n", classification_report(y_test, y_hat, digits=3))